In [1]:

#######################################################################
import re
from metapub import PubMedFetcher, PubMedArticle, pubmedcentral
import logging
import time
import numpy as np
import pandas as pd
import requests
import json
import csv
import os as os
import urllib.parse
from time import sleep
from tqdm import tqdm  
from datetime import datetime, timedelta, date
from functools import partial
from dateutil.relativedelta import relativedelta
from urllib.parse import urlparse, parse_qs
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type, stop_after_attempt, wait_exponential
from multiprocessing.pool import ThreadPool
from typing import List, Iterator, Optional, Dict, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from eutils import EutilsNCBIError, EutilsRequestError
import xml.etree.ElementTree as ET
from urllib.error import HTTPError
import random
#######################################################################


# Decorator 1 
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # Maximum 10 seconds wait.
    wait=wait_fixed(2),  # Wait 400ms between retries
    retry=retry_if_exception_type((Exception,))  # Use a tuple for exceptions
)


fetcher = PubMedFetcher()

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Functions ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

###############Papers DF######################



def fetch_article(pmcid: str) -> PubMedArticle:
    """Fetch a single article from PubMed by PMCID."""
    article = fetcher.article_by_pmcid(pmcid)
    return article

@retry_on_communication_error()
def fetch_articles(pmcids: List[str], *, processes: Optional[int] = 1) -> Iterator[PubMedArticle]:
    """Fetch multiple articles from PubMed in parallel using a thread pool with progress bar."""
    with ThreadPool(processes=processes) as pool:
        # Wrap with tqdm for progress tracking
        for article in tqdm(
            pool.imap_unordered(fetch_article, pmcids),
            total=len(pmcids),
            desc="Fetching PubMed articles",
            unit="article"
        ):
            if article is not None:
                yield article

def fetch_articles_meta(pmids: List[str]) -> pd.DataFrame:
    """Fetch articles and return them as a pandas DataFrame with progress tracking."""
    articles_data = []
    
    # Initialize progress bar for fetching articles
    for article in tqdm(
        fetch_articles(pmids, processes=1),
        total=len(pmids),
        desc="Adding them",
        unit="row"
    ):
        articles_data.append({
            'PMID': article.pmid,
            'PMCID': article.pmc,
            'DOI': article.doi,
            'Title': article.title,
            'Authors': ', '.join(article.authors),
            'Year': article.year,
            'Journal': article.journal,
            'Volume': article.volume,
            'Issue': article.issue,
            'Pages': article.pages,
            'Abstract': article.abstract,
        })

    df = pd.DataFrame(articles_data)
    df['PMCID'] = df['PMCID'].apply(lambda x: f"PMC{x}" if pd.notnull(x) else x)
    return df



#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Examples ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

2025-10-09 15:49:08 littlebeauty numexpr.utils[2770426] INFO Note: NumExpr detected 44 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2025-10-09 15:49:08 littlebeauty numexpr.utils[2770426] INFO NumExpr defaulting to 16 threads.


In [ ]:
help(PubMedFetcher) 


Help on class PubMedFetcher in module metapub.pubmedfetcher:

class PubMedFetcher(metapub.base.Borg)
 |  PubMedFetcher(method='eutils', **kwargs)
 |  
 |  PubMedFetcher (a Borg singleton object backed by an optional SQLite cache)
 |  
 |  An interaction layer for querying via specified method to return PubMedArticle objects.
 |  
 |  Currently available methods: eutils
 |  
 |  Basic Usage:
 |  
 |      fetch = PubMedFetcher()
 |  
 |  To specify a service method (more coming soon):
 |  
 |      fetch = PubMedFetcher('eutils')
 |  
 |  To return an article by querying the service with a known PMID or NCBI Book ID:
 |  
 |      paper = fetch.article_by_pmid('123456')
 |      book = fetch.article_by_pmid('NBK1234')
 |  
 |  Similar methods exist for returning papers by DOI and PM Central id:
 |  
 |      paper = fetch.article_by_doi('10.1038/ng.379')
 |      paper = fetch.article_by_pmcid('PMC3458974')
 |  
 |  Finally, you can search for PMIDs via citation details by using the pmids_for

In [5]:
help(pubmedcentral)

Help on module metapub.pubmedcentral in metapub:

NAME
    metapub.pubmedcentral - An assortment of functions providing access to various web APIs.

DESCRIPTION
    The pubmedcentral.* functions abstract the submission of one of the following
    acceptable IDs to the Pubmed Central ID Conversion API as a lookup to
    get another ID mapping to the same pubmed article:
    
        * doi       Digital Object Identifier
        * pmid      Pubmed ID
        * pmcid     Pubmed Central ID (includes Versioned Identifier)
    
    Available functions:
    
        get_pmid_for_otherid(string)
    
        get_doi_for_otherid(string)
    
        get_pmcid_for_otherid(string)

FUNCTIONS
    get_doi_for_otherid(otherid)
        Use the PMC ID conversion API to attempt to convert either PMID or PMCID to a DOI.
        Returns DOI if successful, or None if there is no 'doi' item in the response.
        
        Note: this method has a very low success rate for retrieving DOIs. Check out the
  

In [4]:
ids = [
    'PMC2193004', 'PMC4122339']

In [5]:
papers_df = fetch_articles_meta(ids) 

Adding them: 100%|██████████| 2/2 [00:00<00:00,  3.87row/s]


In [12]:
pmds = get_pmid_for_otherid(ids)

In [ ]:
################ Download Supplementary Materials ######################
def download_supplementary_materials(oa_pmcids, output_dir="supplementary_materials", format="bioc_xml", delay=1.0):
    """
    Downloads all supplementary materials for a list of PMCIDs using the NCBI BioC API.

    Parameters:
        oa_pmcids (list): List of PMCIDs (e.g., ['PMC1234567', 'PMC2345678']).
        output_dir (str): Directory to save the downloaded supplementary materials.
        format (str): Format of the supplementary materials ('bioc_xml' or 'bioc_json').
        delay (float): Delay in seconds between requests to respect NCBI's rate limits.

    Returns:
        list: List of PMCIDs that had no supplementary materials or failed to download.
    """
    base_url = "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/supplmat.cgi"
    os.makedirs(output_dir, exist_ok=True)

    unsaved_pmcs = []

    for pmcid in oa_pmcids:
        url = f"{base_url}/{format}/{pmcid}/all"
        print(f"Downloading supplementary materials for {pmcid} from {url}")

        try:
            response = requests.get(url)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Failed to download supplementary materials for {pmcid}: {e}")
            unsaved_pmcs.append(pmcid)
            continue

        content_text = response.text.strip()
        if "No result can be found" in content_text:
            print(f"No supplementary materials found for {pmcid}. Skipping save.")
            unsaved_pmcs.append(pmcid)
        else:
            file_extension = "xml" if format == "bioc_xml" else "json"
            file_path = os.path.join(output_dir, f"{pmcid}.{file_extension}")
            with open(file_path, "wb") as file:
                file.write(response.content)
            print(f"Saved supplementary materials for {pmcid} to {file_path}")

        sleep(delay)

    print(f"\nSummary: {len(oa_pmcids)} total PMCIDs processed.")
    print(f"{len(unsaved_pmcs)} had no supplementary materials or failed to download.")
    print(f"{len(oa_pmcids) - len(unsaved_pmcs)} successfully saved.")

    return unsaved_pmcs

In [8]:

def download_pmc_articles(oa_pmcids, output_dir='./Full_text_jsons'):
    """
    Download full-text BioC JSON articles from PubMed Central.
    
    Args:
        oa_pmcids (list): List of PMCID strings to download
        output_dir (str): Output directory for JSON files (default: './Full_text_jsons')
    
    Returns:
        tuple: (success_count, failed_fulltext) where:
            - success_count: Number of successfully downloaded articles
            - failed_fulltext: List of PMCIDs that failed to download
    """
    # Create directory
    os.makedirs(output_dir, exist_ok=True)

    # Initialize lists to track failed downloads
    failed_fulltext = []

    # Initialize progress bar
    with tqdm(oa_pmcids, desc="Downloading articles", unit="article") as pbar:
        for pmcid in pbar:
            # Update description to show current PMCID
            pbar.set_postfix_str(f"PMCID: {pmcid}")
            
            # 1. Download full-text BioC JSON
            fulltext_url = f'https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode'
            try:
                response = requests.get(fulltext_url)
                if response.status_code == 200:
                    with open(f'{output_dir}/{pmcid}.json', 'w', encoding='utf-8') as f:
                        f.write(response.text)
                else:
                    tqdm.write(f"Failed to download full-text for {pmcid} (Status: {response.status_code})")
                    failed_fulltext.append(pmcid)
            except Exception as e:
                tqdm.write(f"Error downloading {pmcid}: {str(e)}")
                failed_fulltext.append(pmcid)
    
    success_count = len(oa_pmcids) - len(failed_fulltext)
    
    # Print summary of failed downloads
    print("\nDownload Summary:")
    print(f"Successfully processed {success_count}/{len(oa_pmcids)} full-text files")
    if failed_fulltext:
        print("\nPMCIDs with full-text download failures:")
        print(failed_fulltext)
    
    return success_count, failed_fulltext



In [ ]:
import logging
import requests
from lxml import etree

def fetch_pmid(pmcid):
    """Fetch PMID from PMCID safely with fallback."""
    # Primary: try Metapub
    try:
        from metapub import PubMedFetcher
        fetch = PubMedFetcher()
        article = fetch.article_by_pmcid(pmcid)
        if article and article.pmid:
            return article.pmid
    except Exception as e:
        logging.warning(f"Metapub failed for {pmcid}: {e}")

    # Fallback: use NCBI ID conversion API directly
    try:
        url = f"https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/?ids={pmcid}&format=json"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        records = data.get("records", [])
        if records and "pmid" in records[0]:
            return records[0]["pmid"]
        else:
            logging.warning(f"No PMID found for {pmcid} in fallback API.")
            return None
    except Exception as e:
        logging.error(f"Fallback API failed for {pmcid}: {e}")
        return None


def get_pmid_for_otherid(pmcid_clean_list):
    """Converts a list of PMCIDs to PMIDs in parallel using multithreading."""
    PMIDs = {}
    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_pmcid = {executor.submit(fetch_pmid, pmcid): pmcid for pmcid in pmcid_clean_list}
        for future in as_completed(future_to_pmcid):
            pmcid = future_to_pmcid[future]
            try:
                pmid = future.result()
                PMIDs[pmcid] = pmid
            except Exception as e:
                logging.error(f"Error processing PMCID {pmcid}: {e}")
                PMIDs[pmcid] = None
    return PMIDs


def extract_text_from_json_to_dataframe(directory: str, section_types: List[str], xml_directory: str = None) -> pd.DataFrame:
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    logger.info(f"Extracting from JSON files in {directory}")

    data_rows = []
    processed_files = 0
    error_files = 0

    for filename in os.listdir(directory):
        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(directory, filename)
        base_filename = os.path.splitext(filename)[0]

        try:
            with open(filepath, 'r', encoding='utf-8') as file:
                try:
                    data = json.load(file)
                except json.JSONDecodeError as e:
                    logger.error(f"Invalid JSON in {filename}: {str(e)}")
                    error_files += 1
                    continue

                try:
                    documents = data[0].get('documents', [])
                    if not documents:
                        logger.warning(f"No documents in {filename}")
                        continue

                    passages = documents[0].get('passages', [])
                    if not passages:
                        logger.warning(f"No passages in {filename}")
                        continue

                    pmc = base_filename
                    pmid = fetch_pmid(pmc) or "NOPMID"

                    entry_serial = 1

                    def get_entry_id():
                        nonlocal entry_serial
                        entry_id = f"{pmid}_{entry_serial:03}"
                        entry_serial += 1
                        return entry_id

                    for passage in passages:
                        infons = passage.get('infons', {})
                        section_type = infons.get('section_type')
                        subtitle = infons.get('type')
                        if section_type in section_types:
                            norm_text = passage.get('text', '').strip()
                            if norm_text:
                                data_rows.append({
                                    "EntryID": get_entry_id(),
                                    "filename": pmc,
                                    "section_type": section_type,
                                    "subtitle": subtitle,
                                    "text": norm_text
                                })

                    if xml_directory:
                        xml_path = os.path.join(xml_directory, base_filename + ".xml")
                        if os.path.isfile(xml_path):
                            xml_df = extract_text_from_bioc_xml(xml_path)
                            for _, row in xml_df.iterrows():
                                text = row['text']
                                if text:
                                    data_rows.append({
                                        "EntryID": get_entry_id(),
                                        "filename": pmc,
                                        "section_type": "SUPPLEMENT",
                                        "subtitle": row.get('type', None),
                                        "text": text.strip()
                                    })

                    processed_files += 1

                except Exception as e:
                    logger.error(f"Error processing {filename}: {str(e)}")
                    error_files += 1

        except IOError as e:
            logger.error(f"Error reading {filename}: {str(e)}")
            error_files += 1

    df = pd.DataFrame(data_rows)

    logger.info(f"Processed {processed_files} files, {error_files} errors")
    logger.info(f"Extracted {len(df)} entries")

    return df


In [ ]:
################################# XML Extraction supp ##########################
def extract_text_from_bioc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    records = []
    filename = os.path.basename(xml_path)
    for doc in root.findall(".//document"):
        document_id = doc.find("id").text if doc.find("id") is not None else None
        passages = doc.findall("passage")

        for passage in passages:
            passage_text = passage.findtext("text", default="")
            infons = {infon.attrib["key"]: infon.text for infon in passage.findall("infon")}
            record = {
                "filename": filename,
                "source": infons.get("source", None),
                "document_id": document_id,
                "type": infons.get("type", None),
                "text": passage_text.strip()
            }
            records.append(record)

    return pd.DataFrame(records)

In [ ]:
##### pipeline #####
def pmc_full_pipeline(
    oa_pmcids: List[str],
    section_types: List[str] = None,
    base_output_dir: str = "./pmc_data",
    download_supplementary: bool = True,
    delay: float = 1.0
) -> pd.DataFrame:
    """Pipeline: Download PMC JSON + supplementary XML, extract text, and save outputs."""

    section_types = section_types or ["ABSTRACT", "INTRO", "METHODS", "DISCUSSION", "TITLE", "CONCL", "FIG", "DISCUSS", "RESULTS"]

    json_dir = os.path.join(base_output_dir, "Full_text_jsons")
    xml_dir = os.path.join(base_output_dir, "supplementary_materials")

    os.makedirs(json_dir, exist_ok=True)
    os.makedirs(xml_dir, exist_ok=True)

    print("\n=== STEP 1: Fetching article metadata ===")
    papers_df = fetch_articles_meta(oa_pmcids)
    papers_df.to_csv(os.path.join(base_output_dir, "Papers.csv"), index=False)
    print(f"Saved article metadata to {os.path.join(base_output_dir, 'Papers.csv')}")

    print("\n=== STEP 2: Downloading full-text JSONs ===")
    success_count, failed_json = download_pmc_articles(oa_pmcids, output_dir=json_dir)

    unsaved_supps = []
    if download_supplementary:
        print("\n=== STEP 3: Downloading supplementary materials ===")
        unsaved_supps = download_supplementary_materials(oa_pmcids, output_dir=xml_dir, format="bioc_xml", delay=delay)

    print("\n=== STEP 4: Extracting content into DataFrame ===")
    df = extract_text_from_json_to_dataframe(json_dir, section_types, xml_directory=(xml_dir if download_supplementary else None))

    df.to_csv(os.path.join(base_output_dir, "Full_text.csv"), index=False)
    print(f"Saved full-text data to {os.path.join(base_output_dir, 'Full_text.csv')}")

    df.attrs['summary'] = {
        'total_pmcids': len(oa_pmcids),
        'json_downloaded': success_count,
        'json_failed': len(failed_json),
        'supp_downloaded': len(oa_pmcids) - len(unsaved_supps) if download_supplementary else 'skipped',
        'supp_failed': len(unsaved_supps) if download_supplementary else 'skipped',
        'entries_extracted': len(df)
    }

    print("\n=== SUMMARY ===")
    for k, v in df.attrs['summary'].items():
        print(f"{k}: {v}")

    return df

In [30]:
df = pmc_full_pipeline(ids, base_output_dir="./paper_files", download_supplementary=True, delay=0.5)


=== STEP 1: Fetching article metadata ===


Adding them: 100%|██████████| 2/2 [00:00<00:00,  4.16row/s]


Saved article metadata to ./paper_files/Papers.csv

=== STEP 2: Downloading full-text JSONs ===



Download Summary:
Successfully processed 2/2 full-text files

=== STEP 3: Downloading supplementary materials ===
No supplementary materials found for PMC2193004. Skipping save.
Saved supplementary materials for PMC4122339 to ./paper_files/supplementary_materials/PMC4122339.xml


2025-10-08 10:59:06 littlebeauty __main__[4084927] INFO Extracting from JSON files in ./paper_files/Full_text_jsons



Summary: 2 total PMCIDs processed.
1 had no supplementary materials or failed to download.
1 successfully saved.

=== STEP 4: Extracting content into DataFrame ===


2025-10-08 10:59:06 littlebeauty __main__[4084927] INFO Processed 2 files, 0 errors
2025-10-08 10:59:06 littlebeauty __main__[4084927] INFO Extracted 122 entries


Saved full-text data to ./paper_files/Full_text.csv

=== SUMMARY ===
total_pmcids: 2
json_downloaded: 2
json_failed: 0
supp_downloaded: 1
supp_failed: 1
entries_extracted: 122


# XML WORKS

In [2]:

def download_xml(oa_pmcids, output_dir="supplementary_materials", format="bioc_json", delay=1.0):
    """
    Downloadsxml for a list of PMCIDs using the NCBI BioC Supplementary Materials API.

    Parameters:
        oa_pmcids (list): List of PMCIDs (e.g., ['PMC1234567', 'PMC2345678']).
        output_dir (str): Directory to save the downloaded supplementary materials.
        format (str): Format of the supplementary materials ('bioc_xml' or 'bioc_json').
        delay (float): Delay in seconds between requests to respect NCBI's rate limits.

    Returns:
        list: List of PMCIDs that had no supplementary materials or failed to download.
    """
    base_url = "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/supplmat.cgi"
    os.makedirs(output_dir, exist_ok=True)

    unsaved_pmcs = []

    for pmcid in oa_pmcids:
        url = f"{base_url}/{format}/{pmcid}/all"
        print(f"Downloading supplementary materials for {pmcid} from {url}")

        try:
            response = requests.get(url)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Failed to download supplementary materials for {pmcid}: {e}")
            unsaved_pmcs.append(pmcid)
            continue

        content_text = response.text.strip()
        if "No result can be found" in content_text:
            print(f"No supplementary materials found for {pmcid}. Skipping save.")
            unsaved_pmcs.append(pmcid)
        else:
            file_extension = "xml" if format == "bioc_xml" else "json"
            file_path = os.path.join(output_dir, f"{pmcid}.{file_extension}")
            with open(file_path, "wb") as file:
                file.write(response.content)
            print(f"Saved supplementary materials for {pmcid} to {file_path}")

        sleep(delay)

    print(f"\nSummary: {len(oa_pmcids)} total PMCIDs processed.")
    print(f"{len(unsaved_pmcs)} had no supplementary materials or failed to download.")
    print(f"{len(oa_pmcids) - len(unsaved_pmcs)} successfully saved.")

    return unsaved_pmcs

In [5]:
pcss = ["PMC3747992",
        "PMC8144423",
        "PMC10830420"]

In [8]:
jsons = download_xml(pcss, output_dir="supplementary_materials", format="bioc_xml", delay=0.4)

Saved supplementary materials for PMC3747992 to supplementary_materials/PMC3747992.xml
Saved supplementary materials for PMC8144423 to supplementary_materials/PMC8144423.xml
Saved supplementary materials for PMC10830420 to supplementary_materials/PMC10830420.xml

Summary: 3 total PMCIDs processed.
0 had no supplementary materials or failed to download.
3 successfully saved.


In [ ]:
def extract_text_from_xml_to_dataframe(directory: str, section_titles: List[str]) -> pd.DataFrame:
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    logger.info(f"Extracting from XML files in {directory}")

    data_rows = []
    seen_texts = set()
    processed_files = 0
    error_files = 0

    section_types_to_exclude = {
            "DATA AVAILABILITY",
            "Conflict of Interest Statement",
            "Declaration of interests",
            "Author contributions",
            "Use of standardised official symbols",
            "List of abbreviations",
            "CONFLICT OF INTEREST",
            "Online content",
            "Funding",
            "Transparency declarations",
            "Availability of data and materials",
            "Competing interests",
            "Electronic supplementary material",
            "Acknowledgement",
            "Acknowledgements",
            "Availability of data and materials",
            "Consent for publication",
            "Consent",
            "Ethics approval",
            "Publisher's Note",
            "Declaration of Competing Interest",
            "Data sharing statement",
            "COI-statement",
            "Conflict of Interest",
            "Authors' contributions",
            "other",
            "Peer review information",
            "ACKNOWLEDGMENTS",
            "Ethical Approval",
            "Ethics approval",
            "Availability of data and materials",
            "Author Contributions",
            "Notes",
            "Usage Notes",
            "DATA DEPOSITION",
            "Author Contributions",
            "Statement of Ethics",
            "Disclosure Statement",
            "Funding Sources",
            "PATIENT CONSENT STATEMENT",
            "AUTHOR CONTRIBUTIONS",
            "FUNDING INFORMATION",
            "Abbreviations",
            "Ethics and Consent",
            "data-availability",
            "AUTHOR CONTRIBUTIONS",
            "ETHICAL APPROVAL",
            "Patient consent",
            "Conflicts of interest",
            "GRANTS",
            "DISCLOSURES",
            "Author contribution statement",
            "Graphical abstract",
            "ACCESSION NUMBERS",
            "Glossary",
            "FUNDING",
            "Publisher’s note",
            "Authors' Contributions",
            "Disclosure",
            "List of Abbreviations",
            "Ethics Statement",
            "competing-interests",
            "Authors' Contributions",
            "datasets",
            "funding-information"

        }

    throwaway_supplementary_paragraphs = {
        "Below is the link to the electronic supplementary material.Supplementary file1 (ZIP 38251 KB)",
        "tx8b00231_si_001.pdf",
        "tx8b00231_si_001.avi."
    }

    throwaway_if_suppinfo = set(throwaway_supplementary_paragraphs)

    supplementary_buffer = []

    def normalize_section_type(section_type):
        section_type = section_type.strip()
        if section_type == "STAR★Methods" or section_type == "STAR⋆METHODS":
            return "Methods"
        if section_type == "Cutting-edge research on therapeutics":
            return "Methods"
        section_type = re.sub(r"^\d+\.\s+", "", section_type)
        if len(section_type.split()) > 3:
            return "Results"
        return section_type

    def flush_supplementary_buffer():
        nonlocal supplementary_buffer
        if len(supplementary_buffer) == 1:
            row = supplementary_buffer[0]
            if row['section_type'].lower().startswith("supplementary") and (
                row['text'] in throwaway_supplementary_paragraphs
                or re.search(r"click here|available at|can be found at|peer review file|reporting summary|supplementary data \\d", row['text'], re.IGNORECASE)
            ):
                supplementary_buffer = []
                return
        if all(re.search(r"click here|available at|can be found at|peer review file|reporting summary|supplementary data \\d", row['text'], re.IGNORECASE) for row in supplementary_buffer):
            supplementary_buffer = []
            return
        data_rows.extend(supplementary_buffer)
        supplementary_buffer = []

    def add_row(section_type, subtitle, text):
        section_type = normalize_section_type(section_type)
        norm_text = re.sub(r'\s+', ' ', text.strip())
        if not norm_text or norm_text in seen_texts:
            return
        if section_type in section_types_to_exclude:
            return
        if section_type.lower().startswith("supplementary") and norm_text in throwaway_supplementary_paragraphs:
            return
        if section_type == "Supplementary Information" and norm_text in throwaway_if_suppinfo:
            return
        if section_type.lower().startswith("supplementary") and subtitle.startswith("title") and len(norm_text.split()) < 5:
            return
        row = {
            "EntryID": get_entry_id(),
            "filename": pmc,
            "section_type": section_type,
            "subtitle": subtitle,
            "text": norm_text
        }
        if section_type.lower().startswith("supplementary"):
            supplementary_buffer.append(row)
        else:
            flush_supplementary_buffer()
            data_rows.append(row)
        seen_texts.add(norm_text)

    for filename in os.listdir(directory):
        if not filename.endswith(".xml"):
            continue

        filepath = os.path.join(directory, filename)
        base_filename = os.path.splitext(filename)[0]

        try:
            tree = ET.parse(filepath)
            root = tree.getroot()

            article_meta = root.find('.//article-meta')
            if article_meta is None:
                logger.warning(f"No article-meta in {filename}")
                continue

            def find_text(path):
                el = article_meta.find(path)
                return el.text.strip() if el is not None and el.text else None

            pmid = find_text("article-id[@pub-id-type='pmid']") or "NOPMID"
            pmc = find_text("article-id[@pub-id-type='pmcid']")
            if not pmc:
                logger.warning(f"Missing PMC ID in {filename}")
                continue

            entry_serial = 6

            def get_entry_id():
                nonlocal entry_serial
                entry_id = f"{pmid}_{entry_serial:03}"
                entry_serial += 1
                return entry_id

            def process_section(sec, parent_title=None, top_section=None):
                sec_title = sec.findtext("title", default="").strip()
                is_top_level = parent_title is None

                if sec_title in section_types_to_exclude:
                    return

                if is_top_level:
                    section_type = sec.get("sec-type") or sec_title or "supplementary-material"
                    current_title = sec_title
                    top_section = section_type
                else:
                    section_type = top_section
                    current_title = sec_title

                if sec_title:
                    add_row(section_type, "title_1", sec_title)

                for elem in sec:
                    if elem.tag == "sec":
                        process_section(elem, parent_title=current_title, top_section=top_section)
                    elif elem.tag == "title":
                        continue
                    elif elem.tag == "p":
                        para_text = ET.tostring(elem, method='text', encoding='unicode').strip()
                        add_row(section_type, "paragraph", para_text)
                    elif elem.tag == "fig":
                        process_figure(elem, section_type)
                    elif elem.tag == "supplementary-material":
                        process_supplementary(elem)

            def process_figure(fig, section_type):
                fig_label = fig.findtext("label", default="")
                fig_caption = fig.find("caption")
                if fig_caption is not None:
                    caption_parts = [re.sub(r'\s+', ' ', ET.tostring(child, method='text', encoding='unicode')).strip() for child in fig_caption if child.text or len(child)]
                    caption_text = ' '.join(caption_parts).strip()
                    if caption_text:
                        label_text = f"{fig_label}: {caption_text}" if fig_label else caption_text
                        section_type_override = "Extended Data" if 'Extended Data' in fig_label else section_type
                        add_row(section_type_override, "fig_caption", label_text)

            def process_supplementary(elem):
                label = elem.findtext(".//label", default="").strip()
                caption_elem = elem.find(".//caption")
                if caption_elem is not None:
                    caption_parts = [ET.tostring(p, method='text', encoding='unicode').strip() for p in caption_elem.findall(".//p")]
                    caption_text = ' '.join(caption_parts).strip()
                    full_text = f"{label}: {caption_text}" if label else caption_text
                    add_row("supplementary-material", "paragraph", full_text)

            body = root.find(".//body")
            if body is not None:
                seen_first_section = False
                for elem in body:
                    if elem.tag == "sec":
                        process_section(elem)
                        seen_first_section = True
                    elif elem.tag == "p":
                        para_text = ET.tostring(elem, method='text', encoding='unicode').strip()
                        inferred_section = "Introduction" if not seen_first_section else "Results"
                        add_row(inferred_section, "paragraph", para_text)

            back = root.find(".//back")
            if back is not None:
                for sec in back.findall(".//sec"):
                    sec_title = sec.findtext("title", default="").strip()
                    if sec_title in section_types_to_exclude:
                        continue
                    process_section(sec)

            ref_list = root.find(".//ref-list")
            if ref_list is not None:
                for ref in ref_list.findall("ref"):
                    citation_texts = []
                    for elem in ref.iter():
                        if elem.text and elem.tag not in {"ref"}:
                            citation_texts.append(elem.text.strip())
                    full_citation = re.sub(r'\s+', ' ', " ".join(citation_texts).strip())
                    if full_citation:
                        add_row("References", "paragraph", full_citation)

            flush_supplementary_buffer()
            processed_files += 1

        except Exception as e:
            logger.error(f"Error processing {filename}: {str(e)}")
            error_files += 1

    df = pd.DataFrame(data_rows, columns=["EntryID", "filename", "section_type", "subtitle", "text"])
    logger.info(f"Processed {processed_files} files, {error_files} errors")
    logger.info(f"Extracted {len(df)} entries")
    return df
